In [0]:
from pyspark.sql.functions import col
df_silver = spark.table("marathos_catalog.marathon_silver.silver_cleaned")


In [0]:
# 1. Antal rader och kolumner
print(f"\n1. Antal rader: {df_silver.count():,}")
print(f"2. Antal kolumner: {len(df_silver.columns)}")

In [0]:
from pyspark.sql.functions import col

print("kontroll av rensing\n")

# Vi kan inte kolla raw eftersom den är borta, men vi kan kolla att alla tider är rimliga

# 2. Kolla att inga 0 värden finns kvar
has_zero = df_silver.filter(col("athlete_performance") == 0).count()
print(f"Rader med athlete_performance = 0 (borde vara 0): {has_zero}")

# 3. Kolla att inga negativa värden finns
has_negative = df_silver.filter(col("athlete_performance") < 0).count()
print(f"Rader med negativ tid (borde vara 0): {has_negative}")

# 4. Kolla att event_id finns och är unikt
unique_events = df_silver.select("event_id").distinct().count()
total_rows = df_silver.count()
print(f"Antal unika event_id: {unique_events:,} (av {total_rows:,} rader)")

# 5. Kolla att inga event_id är null
null_events = df_silver.filter(col("event_id").isNull()).count()
print(f"Rader med null event_id (borde vara 0): {null_events}")

if has_zero == 0 and has_negative == 0 and null_events == 0:
    print("\nAll rensning lyckades!")
else:
    print("\nDet finns problem i rensningen!")

In [0]:
#Visa schema
print("\n3. Schema:")
df_silver.printSchema()

In [0]:
unique_events = df_silver.select("event_id").distinct().count()
print(f"\n6. Antal unika event: {unique_events}")

In [0]:
print("Mest populära distanser")

distance_dist = df_silver.groupBy("event_distance").count() \
    .orderBy("count", ascending=False) \
    .limit(10)

distance_dist = distance_dist.withColumnRenamed("event_distance", "distans")
distance_dist = distance_dist.withColumnRenamed("count", "antal_deltagare")

distance_dist.display()

In [0]:
# Topp 10 
print(" Topp 10 länder med flest deltgare:")
df_silver.groupBy("athlete_country").count().orderBy("count", ascending=False).limit(10).display()